# GK-2A 12:00~14:00 증분형 테이블 파이프라인

기존 Phase 1/2 결과를 재사용하지 않는 새 파이프라인입니다. 현재 Drive에 있는 NC만 읽고, 없는 채널·시각은 NaN으로 둡니다. 나중에 API 제한이 풀리면 **결측 파일만** 받은 뒤 같은 CSV를 원자적으로 다시 써서 해당 칸을 자동으로 채웁니다.

범위: 2019~2025년, 8월 24~30일, 12:00~14:00 KST, 10분 간격, GK-2A LE1B/KO 16채널. 위·경도·고도는 저장소의 대회 공식 `station_list.csv`만 사용합니다.

In [ ]:
# 1. Google Drive 마운트 — 기존 /content/drive와 충돌하지 않는 빈 경로를 선택
import os
from pathlib import Path
from google.colab import drive

candidates = [Path('/content/gdrive_sme_incremental'), Path('/content/gdrive_sme_incremental_2')]
mounted = next((p for p in candidates if os.path.ismount(p)), None)
if mounted is not None:
    MOUNT_POINT = mounted
    print('이미 마운트됨:', MOUNT_POINT)
else:
    MOUNT_POINT = next((p for p in candidates if not p.exists() or not any(p.iterdir())), None)
    if MOUNT_POINT is None:
        raise RuntimeError('후보 마운트 폴더에 일반 파일이 있습니다. 런타임을 재시작한 뒤 이 셀부터 실행하세요.')
    drive.mount(str(MOUNT_POINT), force_remount=False)
print('MOUNT_POINT =', MOUNT_POINT)

In [ ]:
# 2. 경로와 실행 범위
import subprocess, sys

BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_DIR = Path('/content/SME_DATA_incremental')

# 다른 폴더에 데이터를 두었다면 따옴표 안에 절대경로를 넣으세요. 비워두면 기본값을 씁니다.
DATA_ROOT_OVERRIDE = ''
default_root = MOUNT_POINT / 'MyDrive/SME_DATA/processed_station_features/shortterm_12to14_data'
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else default_root
OUTPUT_DIR = DATA_ROOT / 'incremental_12to14_tables'

YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
MAX_PROCESS_FILES = 0  # 0=현재 존재하는 새/변경 NC 전부. 중단돼도 캐시부터 이어집니다.

if not (DATA_ROOT / 'raw_gk2a').exists():
    raise FileNotFoundError(f'raw_gk2a 폴더를 찾지 못했습니다: {DATA_ROOT / "raw_gk2a"}\nDATA_ROOT_OVERRIDE를 실제 경로로 바꾸세요.')
print('DATA_ROOT =', DATA_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# 3. 새 코드 받기 및 의존성 설치
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'xarray', 'h5netcdf', 'netCDF4', 'requests'], check=True)
print('준비 완료:', REPO_DIR)

In [ ]:
# 4. 공통 실행 함수
def run_live(cmd):
    print('$', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        list(map(str, cmd)), cwd=REPO_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
    if returncode:
        raise RuntimeError(f'실행 실패 (exit code {returncode})')

def pipeline_cmd(phase):
    return [
        sys.executable, '-u', 'scripts/incremental_shortterm_pipeline.py',
        '--phase', phase, '--data-root', DATA_ROOT, '--output-dir', OUTPUT_DIR,
        '--station-list', REPO_DIR / 'data/metadata/station_list.csv',
        '--years', *YEARS, '--start-mmdd', '08-24', '--end-mmdd', '08-30',
        '--start-time', '12:00', '--end-time', '14:00', '--step-minutes', '10',
        '--max-process-files', MAX_PROCESS_FILES,
    ]

In [ ]:
# 5. 현재 보유 NC 현황만 빠르게 확인
run_live(pipeline_cmd('inventory'))

In [ ]:
# 6. 현재 보유 NC로 테이블 생성
# 최초에는 NC를 읽어 station feature cache를 만드므로 오래 걸릴 수 있습니다.
# Colab이 중단되어도 같은 셀을 다시 실행하면 완료된 캐시는 건너뜁니다.
run_live(pipeline_cmd('sync'))

In [ ]:
# 7. 결과 확인
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / 'incremental_summary_2019to2025.csv'
missing_path = OUTPUT_DIR / 'missing_nc_inventory_2019to2025.csv'
display(pd.read_csv(summary_path))
missing = pd.read_csv(missing_path)
print('남은 결측 NC:', len(missing))
display(missing.head(20))
print('LSTM long CSV:', OUTPUT_DIR / 'shortterm_long_2019to2025.csv')
print('wide CSV:', OUTPUT_DIR / 'shortterm_wide_2019to2025.csv')

## API 제한이 풀린 뒤 실행

아래 셀은 `source_inventory.csv`에서 아직 없는 NC만 요청합니다. 받은 파일은 `raw_gk2a/연도/월/일`에 저장되고, 이어서 새 파일만 캐시에 반영한 후 **동일한 long/wide CSV를 자동 갱신**합니다. 실패한 요청은 NaN 상태로 남으므로 기존 정상값을 지우지 않습니다.

In [ ]:
# 8. 결측 NC만 재수집하고 테이블 자동 갱신
import getpass
try:
    from google.colab import userdata
    KMA_API_KEY = userdata.get('KMA_API_KEY')
except Exception:
    KMA_API_KEY = None
if not KMA_API_KEY:
    KMA_API_KEY = getpass.getpass('KMA API authKey (화면에 표시되지 않음): ').strip()
if not KMA_API_KEY:
    raise ValueError('KMA_API_KEY가 비어 있습니다.')
os.environ['KMA_API_KEY'] = KMA_API_KEY

MAX_DOWNLOADS = 200  # 1회 최대 요청 수; 필요하면 낮추세요.
retry_cmd = pipeline_cmd('retry-and-sync') + [
    '--max-downloads', MAX_DOWNLOADS, '--request-interval', 0.5,
    '--max-retries', 3, '--api-key-env', 'KMA_API_KEY',
]
run_live(retry_cmd)
os.environ.pop('KMA_API_KEY', None)
print('재수집 및 자동 갱신 완료. 7번 셀에서 남은 결측 수를 다시 확인하세요.')

In [ ]:
# 9. NC를 Drive에 수동으로 추가한 경우: 다운로드 없이 표만 다시 동기화
run_live(pipeline_cmd('sync'))